# EARAS/REXA — Exploratory Data Analysis

Quick look at the sample dataset (`data/raw/sample_questions.json` and
`data/raw/sample_answers.json`): question/course distribution, star rating
balance, sentence-role balance, and concept coverage.

Run this notebook with its working directory set to `ml/notebooks/` (the
default when opened directly in Jupyter/VS Code/Cursor).

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../../data")
questions = json.loads((DATA_DIR / "raw" / "sample_questions.json").read_text(encoding="utf-8"))
answers = json.loads((DATA_DIR / "raw" / "sample_answers.json").read_text(encoding="utf-8"))

print(f"Questions: {len(questions)}")
print(f"Answers: {len(answers)}")

questions_df = pd.DataFrame(questions)
answers_df = pd.DataFrame(answers)
questions_df[["id", "title", "course"]]

## Star rating & quality tier distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(x="stars", data=answers_df, order=sorted(answers_df["stars"].unique()), ax=axes[0])
axes[0].set_title("Star rating distribution")

sns.countplot(
    x="quality_tier",
    data=answers_df,
    order=["excellent", "good", "average", "weak", "poor"],
    ax=axes[1],
)
axes[1].set_title("Authoring quality tier distribution")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

answers_df["stars"].describe()

## Sentence role balance

Flatten every answer's `sentence_roles` to see how balanced the five role
classes are (important for Module 1 training — classes should not be wildly
imbalanced given the small sample size).

In [ ]:
role_counter = Counter()
support_label_counter = Counter()
concept_coverage_ratios = []

for a in answers:
    for sr in a["sentence_roles"]:
        role_counter[sr["role"]] += 1
    for pair in a["support_pairs"]:
        support_label_counter[pair["label"]] += 1
    if a["concepts"]:
        concept_coverage_ratios.append(len(a["concepts_present"]) / len(a["concepts"]))

print("Sentence role counts:", dict(role_counter))
print("Support/contradiction label counts:", dict(support_label_counter))

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=list(role_counter.keys()), y=list(role_counter.values()), ax=ax)
ax.set_title("Sentence role counts across all sample answers")
plt.show()

pd.Series(concept_coverage_ratios, name="coverage_ratio").describe()

## Observations

- Stars span the full 1-5 range with a reasonable spread across tiers,
  which is necessary for the regression-based star-prediction module to
  learn a meaningful signal.
- `Evidence` and `Claim` are the most common sentence roles by construction
  (every quality tier includes a claim, and most include at least one
  evidence sentence); `Conclusion` and `Other` are comparatively rarer,
  which is worth monitoring as more real data is collected.
- Concept coverage ratio correlates strongly with quality tier and star
  rating by construction — this is exactly the signal the star-prediction
  and concept-coverage modules are trained to pick up on.
- This dataset is intentionally small (50 answers) and synthetic. Treat all
  metrics in `ml/checkpoints/*/metrics.json` as a pipeline smoke test, not a
  claim of real-world model quality.